<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.5-structured-output-and-tool-use/notebooks/exercises-3.5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.5 — Structured Output & Tool Use
## Practice Exercises

Netsetos GenAI Course v2.0 | Module 3: Prompt Engineering


In [ ]:
# Setup
!pip install pydantic google-generativeai openai anthropic instructor -q
import json, re, os
from pydantic import BaseModel, field_validator, ValidationError


## Exercise 1: Prompt-Based JSON Extraction


In [ ]:
# Extract business card info using prompt tricks
PROMPT = """Extract the following from this business card text.
Return ONLY valid JSON, no markdown, no explanation.
Schema: {"name": str, "email": str, "phone": str|null, "company": str}

Text: Priya Sharma, CTO at TechNova Solutions, priya@technova.in, +91-98765-43210
"""

# TODO: Call your preferred LLM and parse the response
# Hint: Use json.loads() with try/except for error handling
# Test edge cases: missing phone, multiple emails, non-English name


## Exercise 2: XML Multi-Section Parser


In [ ]:
# Parse multi-section XML output
PROMPT = """Analyze this review and return in XML format:
<summary>2-sentence summary</summary>
<entities>JSON list of entities mentioned</entities>
<sentiment>positive/negative/neutral</sentiment>

Review: The new Tata Nexon EV Max has excellent range of 437km. 
However, the charging infrastructure in tier-2 cities is still poor.
"""

def parse_xml_sections(text):
    """Extract content from XML tags using regex."""
    sections = {}
    for tag in ["summary", "entities", "sentiment"]:
        match = re.search(f"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
        sections[tag] = match.group(1).strip() if match else None
    return sections

# TODO: Call LLM and parse with parse_xml_sections()


## Exercise 3: Pydantic Validation Pipeline


In [ ]:
from enum import Enum

class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"

class ProductReview(BaseModel):
    product_name: str
    sentiment: Sentiment
    rating: int
    review_text: str
    
    @field_validator('rating')
    @classmethod
    def rating_range(cls, v):
        if v < 1 or v > 5:
            raise ValueError('Rating must be 1-5')
        return v
    
    @field_validator('review_text')
    @classmethod
    def min_review_length(cls, v):
        if len(v) < 10:
            raise ValueError('Review must be at least 10 characters')
        return v

# TODO: Build retry loop that feeds ValidationError back to LLM
# Hint: catch ValidationError, format as string, append to messages


## Exercise 4: Provider Comparison — Structured Outputs


In [ ]:
# OpenAI strict mode
from openai import OpenAI

class MovieReview(BaseModel):
    title: str
    director: str
    year: int
    rating: float
    genres: list[str]

# client = OpenAI()
# completion = client.chat.completions.parse(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": "Review: Oppenheimer by Christopher Nolan (2023) - brilliant 9/10 drama/thriller"}],
#     response_format=MovieReview,
# )
# print(completion.choices[0].message.parsed)

# TODO: Implement same extraction with Claude output_config and Gemini responseSchema
# Compare: reliability, latency, token cost


## Exercise 5: Function Calling Weather Agent


In [ ]:
# Define tools for a weather agent
def get_weather(city: str) -> dict:
    """Get current weather for a city."""
    mock_data = {"Delhi": "42°C Sunny", "Mumbai": "32°C Humid", "Bangalore": "28°C Cloudy"}
    return {"city": city, "weather": mock_data.get(city, "25°C Clear")}

def get_forecast(city: str, days: int) -> dict:
    """Get weather forecast for N days."""
    return {"city": city, "days": days, "forecast": [f"Day {i+1}: {25+i}°C" for i in range(days)]}

def convert_temp(value: float, from_unit: str, to_unit: str) -> dict:
    """Convert temperature between Celsius and Fahrenheit."""
    if from_unit == "C" and to_unit == "F":
        return {"result": value * 9/5 + 32, "unit": "F"}
    elif from_unit == "F" and to_unit == "C":
        return {"result": (value - 32) * 5/9, "unit": "C"}
    return {"error": f"Unknown conversion: {from_unit} to {to_unit}"}

# TODO: Define these as tool schemas for your preferred provider
# Test: "What's the weather in Delhi?" → get_weather
# Test: "5-day forecast for Mumbai" → get_forecast
# Test: "Convert 100F to Celsius" → convert_temp
# Test: "Hello, how are you?" → no tool


## Exercise 6: instructor Multi-Provider


In [ ]:
# import instructor

class JobPosting(BaseModel):
    title: str
    company: str
    location: str
    salary_min: int
    salary_max: int
    remote: bool
    
    @field_validator('salary_max')
    @classmethod
    def max_gt_min(cls, v, info):
        if 'salary_min' in info.data and v < info.data['salary_min']:
            raise ValueError('salary_max must be >= salary_min')
        return v

# client = instructor.from_provider("openai/gpt-4o-mini")
# job = client.chat.completions.create(
#     response_model=JobPosting,
#     messages=[{"role": "user",
#         "content": "Hiring: Senior ML Engineer at Razorpay Bangalore, 30-50 LPA, hybrid"}],
#     max_retries=3,
# )
# print(job.model_dump_json(indent=2))

# TODO: Switch to Claude and Gemini with one-line change
# Measure: how many retries each provider needs


## Exercise 7: GSTIN/PAN Cross-Validator


In [ ]:
import re

VALID_STATE_CODES = {f"{i:02d}" for i in range(1, 38)}

class IndianBusinessEntity(BaseModel):
    company_name: str
    gstin: str | None = None
    pan: str | None = None
    
    @field_validator('gstin')
    @classmethod
    def validate_gstin(cls, v):
        if v and not re.match(
            r'^([0][1-9]|[1-2][0-9]|[3][0-7])[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z]$', v
        ):
            raise ValueError(f'Invalid GSTIN format: {v}')
        if v and v[:2] not in VALID_STATE_CODES:
            raise ValueError(f'Invalid state code: {v[:2]}')
        return v
    
    @field_validator('pan')
    @classmethod
    def validate_pan(cls, v):
        if v and not re.match(r'^[A-Z]{3}[ABCFGHLJPTF][A-Z][0-9]{4}[A-Z]$', v):
            raise ValueError(f'Invalid PAN: {v}. 4th char must be holder type.')
        return v

# Test cases
test_cases = [
    {"company_name": "TCS", "gstin": "27AAACT2727Q1ZV", "pan": "AAACT2727Q"},  # Valid
    {"company_name": "Test", "gstin": "99AAACT2727Q1ZV", "pan": "AAACT2727Q"},  # Invalid state
    {"company_name": "Test", "gstin": "27AAACT2727Q1ZV", "pan": "BBBBB1234C"},  # Mismatch
]

for tc in test_cases:
    try:
        entity = IndianBusinessEntity(**tc)
        print(f"✅ {tc['company_name']}: Valid")
    except ValidationError as e:
        print(f"❌ {tc['company_name']}: {e.errors()[0]['msg']}")


## Exercise 8: GST Invoice Extractor


In [ ]:
class GSTLineItem(BaseModel):
    description: str
    hsn_code: str
    quantity: int
    rate: float
    taxable_value: float
    cgst: float = 0.0
    sgst: float = 0.0
    igst: float = 0.0

class GSTInvoice(BaseModel):
    invoice_number: str
    invoice_date: str
    supplier_gstin: str
    buyer_gstin: str
    line_items: list[GSTLineItem]
    total_taxable: float
    total_tax: float
    grand_total: float
    
    @field_validator('line_items')
    @classmethod
    def validate_tax_type(cls, items):
        """CGST+SGST for intra-state, IGST for inter-state"""
        for item in items:
            has_cgst_sgst = item.cgst > 0 or item.sgst > 0
            has_igst = item.igst > 0
            if has_cgst_sgst and has_igst:
                raise ValueError(
                    f"Item '{item.description}': Cannot have both CGST/SGST and IGST"
                )
        return items

# TODO: Use instructor to extract from sample invoice text
# Sample: "Invoice #INV-2026-001, Date: 15-Mar-2026
# From: Infosys Ltd (GSTIN: 29AABCI1234F1Z5)
# To: Wipro Ltd (GSTIN: 29AABCW5678G1ZP)
# Item: Cloud Consulting, HSN 998314, 100 hrs @ ₹5000, CGST 9%, SGST 9%"


---
## Recap
You've practiced the complete structured output stack: prompt tricks → Pydantic → native APIs → instructor → India-specific extraction. The hierarchy is clear: **always use constrained decoding (Tier 1) in production, with Pydantic as your safety net.**
